In [52]:
import pandas as pd
import numpy as np
import pickle

# Getting familiar with the database

In this notebook we will present the preliminary preprocessing steps for loading and starting to use our database.

## Data loading and preprocessing

We are going to load the database and access the information we are going to need for our recommender system. In particular, we will specify which hospital's data we want to use and the antibioatics of interest.

In [53]:
"""""
db = pd.read_excel('Data/DB_conjunta.xlsx')

centre = 'HGM' # Indicate the center you want to analyse
db = db[db['Centro'] == centre]
db = db.iloc[:,[0,14,16,18,20,22,24,26,28,30,32,34,36,38,40,42]] # Select the columns with the antibiotics
# Replace the values for numbers and NaN. 1: Resistant or Intermediate, 0: Sensitive, NaN: not tested
db.replace({"R": 1, "S": 0, "I": 1, '-': np.nan, '': np.nan}, inplace=True)
#  Remove rows with all NaN values
db.dropna(how='all', subset=db.columns[1:], inplace=True)
# Remove rows with repeated "Número de muestra" values, keeping the first occurrence
db.drop_duplicates(subset='Número de muestra', keep='first', inplace=True)
db
"""


with open('/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/COMBINED_MARISMA_DRIAMS.pkl', 'rb') as f:
    payload = pickle.load(f)

X = payload["data"]              # MALDI spectra (n_samples, 6000)
y_species = payload["label"]     # species
amr = payload["amr"]             # (n_samples, n_antibiotics)
antibiotics = payload["antibiotics"]

Here we modify the matrix to have a configuration typically used in recommender systems. The _sample_id_ colum represents the sample's identifier, the _item_id_ column representes the antibiotic name and the _resistance_ column represents whether the sample is resistant to the antibioatic or not.

## Filer species

In [54]:
species = "Klebsiella_Pneumoniae"  # cambia según quieras

mask = (y_species == species)

X = X[mask]
amr = amr[mask]
y_species = y_species[mask]

In [55]:
"""""
# Reshape the data to recomended format
db_rec = db.melt(id_vars=[db.columns[0]], var_name='item_id', value_name='resistance').rename(columns={db.columns[0]: 'sample_id'})
# Remove rows with NaN values
db_rec = db_rec.dropna()
db_rec
"""

rows = []

for i in range(X.shape[0]):
    for j in range(amr.shape[1]):
        value = amr[i, j]

        # saltar missing SOLO si realmente lo es
        if value is None:
            continue

        if isinstance(value, float) and np.isnan(value):
            continue

        if value == -1:  
            continue

        rows.append({
            "sample_id": i,
            "item_id": antibiotics[j],
            "resistance": int(value),
            "maldi": X[i],
            "sample_type": y_species[i]
        })
db_rec = pd.DataFrame(rows)
print("Rows:", len(db_rec))
print("Columns:", db_rec.columns)


Rows: 340703
Columns: Index(['sample_id', 'item_id', 'resistance', 'maldi', 'sample_type'], dtype='str')


## Adding MALDI-TOF spectrums to the data
Let's now load the associated MALDI-TOF spectrum and add them to the dataframe.

In [56]:
"""""
# Cargar el archivo pkl
with open('Data/gm_data_processed.pkl', 'rb') as f:
    data = pickle.load(f)
df = pd.DataFrame(data['full'])
df
"""

'""\n# Cargar el archivo pkl\nwith open(\'Data/gm_data_processed.pkl\', \'rb\') as f:\n    data = pickle.load(f)\ndf = pd.DataFrame(data[\'full\'])\ndf\n'

Now we can combine these with the original dataframe, having that the identifier in _Nº Micro_ corresponds to the identifier _sample_id_ on the previous dataframe.

In [57]:
"""""
db_rec = db_rec.merge(df[['Nº Micro', 'maldi', 'Muestra']], left_on='sample_id', right_on='Nº Micro', how='left')
db_rec.drop(columns=['Nº Micro'], inplace=True)
db_rec
"""

'""\ndb_rec = db_rec.merge(df[[\'Nº Micro\', \'maldi\', \'Muestra\']], left_on=\'sample_id\', right_on=\'Nº Micro\', how=\'left\')\ndb_rec.drop(columns=[\'Nº Micro\'], inplace=True)\ndb_rec\n'

Let's remove the entries that do not have an associated MALDI-TOF.

In [58]:
"""""
db_rec.dropna(subset=['maldi'], inplace=True)
db_rec
"""

'""\ndb_rec.dropna(subset=[\'maldi\'], inplace=True)\ndb_rec\n'

The neural network requires having numeric ids, so we are going to encode them.

In [59]:
print(db_rec.head())
print(db_rec.columns)
print(len(db_rec))

   sample_id        item_id  resistance  \
0          0       Amikacin           0   
1          0       Cefepime           1   
2          0      Cefoxitin           1   
3          0  Ciprofloxacin           1   
4          0       Colistin           0   

                                               maldi            sample_type  
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  Klebsiella_Pneumoniae  
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  Klebsiella_Pneumoniae  
2  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  Klebsiella_Pneumoniae  
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  Klebsiella_Pneumoniae  
4  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  Klebsiella_Pneumoniae  
Index(['sample_id', 'item_id', 'resistance', 'maldi', 'sample_type'], dtype='str')
340703


In [60]:
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
"""""
enc_items = LabelEncoder()
db_rec['item_id'] = enc_items.fit_transform(db_rec['item_id'])
enc_meta = LabelEncoder()
db_rec['Muestra'] = enc_meta.fit_transform(db_rec['Muestra'])
db_rec.rename(columns={'Muestra': 'sample_type'}, inplace=True)
db_rec
"""


from sklearn.preprocessing import LabelEncoder

enc_items = LabelEncoder()
db_rec['item_id'] = enc_items.fit_transform(db_rec['item_id'])

enc_meta = LabelEncoder()
db_rec['sample_type'] = enc_meta.fit_transform(db_rec['sample_type'])

This way we have a database with the following characteristics:

In [61]:
num_samples = len(db_rec['sample_id'].unique())
num_items = len(db_rec['item_id'].unique())
feature_dim = len(db_rec['maldi'][0])
type_dim = len(db_rec['sample_type'].unique())

print('Number of samples: ', num_samples)
print('Number of items: ', num_items)
print('Spectrum dimension: ', feature_dim)
print('Sample type dimension: ', type_dim)
print('Number of ratings: ', len(db_rec))

Number of samples:  19265
Number of items:  48
Spectrum dimension:  6000
Sample type dimension:  1
Number of ratings:  340703


In [62]:
db_rec['sample_id'] = db_rec['sample_id'].astype(int)

### Filter out antibiotics with few samples

In [63]:
counts = db_rec.groupby('item_id')['resistance'].count()
valid_items = counts[counts > 50].index
db_rec = db_rec[db_rec['item_id'].isin(valid_items)]

In [64]:
feature_dim = len(db_rec['maldi'].iloc[0])  # debería ser 6000

# Neural Collaborative Filtering (NCF)

Let's test our data using NCF as a recommender system. We are going to make partitions based on unique maldi-tofs
Now that eveything is ready to go, let's train our model for each partition and measure the overall performance.

In [65]:
def data_leakage(X_tr, X_tst, X_val):
    # Verify that all samples are being used
    tr_ids = len(X_tr['sample_id'].unique())
    tst_ids = len(X_tst['sample_id'].unique())
    val_ids = len(X_val['sample_id'].unique())
    assert tr_ids + tst_ids + val_ids == num_samples, "Error! Not all samples are being used."

    # Verify that there is no data leakage between tr, tst and val
    assert len(set(X_tr['sample_id'].unique()) & set(X_tst['sample_id'].unique())) == 0, "Error! There are commont samples between tr and tst."
    assert len(set(X_tr['sample_id'].unique()) & set(X_val['sample_id'].unique())) == 0, "Error! There are commont samples between tr and val."
    assert len(set(X_tst['sample_id'].unique()) & set(X_val['sample_id'].unique())) == 0, "Error! There are commont samples between val and tst."

    print("No data leakage detected.")

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl

from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from pytorch_lightning.callbacks.early_stopping import EarlyStopping

from lib.NCF import NCF
from lib.RecDataset import RecDataset


# =========================
# CONFIG
# =========================
model_name = 'NCF'
maldi_encoder = 'CNN'   # 'MLP' or 'CNN'
meta_flag = True

type_str = '_metadata' if meta_flag else ''
filename = f"{model_name}{type_str}_{maldi_encoder}"

n_splits = 5

embedding_dim_samples = 30
embedding_dim_items = 15
embedding_dim_metadata = 10
hidden_dim_samples = [500]
patience = 15
batch_size = 32
max_epochs = 300

os.makedirs("Models", exist_ok=True)


# =========================
# OPTIONAL FILTER:
# keep only antibiotics with enough samples
# and both classes present
# =========================
def filter_valid_antibiotics(df, min_samples=50):
    valid_items = []
    for item in df['item_id'].unique():
        subset = df[df['item_id'] == item]['resistance']
        if len(subset) >= min_samples and subset.nunique() > 1:
            valid_items.append(item)
    return df[df['item_id'].isin(valid_items)].copy()


# Apply filter if you want
db_rec = filter_valid_antibiotics(db_rec, min_samples=50).reset_index(drop=True)

from sklearn.preprocessing import LabelEncoder

# Re-encode item_id after filtering so indices go from 0 to n_items-1
enc_items = LabelEncoder()
db_rec['item_id'] = enc_items.fit_transform(db_rec['item_id'])

# Re-encode sample_type too, to be safe
enc_meta = LabelEncoder()
db_rec['sample_type'] = enc_meta.fit_transform(db_rec['sample_type'])

# =========================
# BASIC INFO
# =========================
num_items = db_rec['item_id'].nunique()
num_meta = db_rec['sample_type'].nunique()
num_samples = db_rec['sample_id'].nunique()

print("Number of unique samples:", num_samples)
print("Number of antibiotics:", num_items)
print("Number of metadata classes:", num_meta)
print("Total rows:", len(db_rec))


# =========================
# SCORES
# =========================
categories = ['tr', 'val', 'tst']
metrics = ['acc', 'auc']

if not os.path.exists(filename + '_scores.pkl'):
    scores_dict = {
        metric: {
            fold: {category: 0 for category in categories}
            for fold in range(n_splits)
        }
        for metric in metrics
    }
else:
    with open(filename + '_scores.pkl', 'rb') as file:
        scores_dict = pickle.load(file)


def update_score(metric, fold, category, value):
    if metric in scores_dict and fold in scores_dict[metric] and category in scores_dict[metric][fold]:
        if isinstance(value, np.ndarray):
            value = value.tolist()
        scores_dict[metric][fold][category] = value
    else:
        print(f"Invalid metric, fold, or category: {metric}, {fold}, {category}")


# =========================
# DATA LEAKAGE CHECK
# =========================
def data_leakage(X_tr, X_tst, X_val):
    tr_ids = len(X_tr['sample_id'].unique())
    tst_ids = len(X_tst['sample_id'].unique())
    val_ids = len(X_val['sample_id'].unique())

    assert tr_ids + tst_ids + val_ids == num_samples, "Error! Not all samples are being used."
    assert len(set(X_tr['sample_id'].unique()) & set(X_tst['sample_id'].unique())) == 0, "Error! Common samples between train and test."
    assert len(set(X_tr['sample_id'].unique()) & set(X_val['sample_id'].unique())) == 0, "Error! Common samples between train and val."
    assert len(set(X_tst['sample_id'].unique()) & set(X_val['sample_id'].unique())) == 0, "Error! Common samples between test and val."

    print("No data leakage detected.")


# =========================
# CREATE FOLDS BY SAMPLE_ID
# =========================
groups = db_rec.groupby('sample_id')
folds = [[] for _ in range(n_splits)]

for i, (_, group) in enumerate(groups):
    folds[i % n_splits].append(group)

fold_dfs = [pd.concat(fold).reset_index(drop=True) for fold in folds]


# =========================
# STORE AUC PER ANTIBIOTIC
# =========================
antibiotic_auc = {i: [] for i in range(num_items)}


# =========================
# TRAIN / EVAL LOOP
# =========================
for fold in range(n_splits):
    print(f"\n========== Fold {fold} ==========")

    X_tst = fold_dfs[fold]
    X_val = fold_dfs[(fold + 1) % n_splits]
    training_indices = [j for j in range(n_splits) if j != fold and j != (fold + 1) % n_splits]
    X_tr = pd.concat([fold_dfs[j] for j in training_indices]).reset_index(drop=True)

    data_leakage(X_tr, X_tst, X_val)

    # Data arrays
    maldis_tr = np.stack(X_tr['maldi'].values)
    maldis_val = np.stack(X_val['maldi'].values)
    maldis_tst = np.stack(X_tst['maldi'].values)

    meta_tr = X_tr['sample_type'].values
    meta_val = X_val['sample_type'].values
    meta_tst = X_tst['sample_type'].values

    drugs_tr = X_tr['item_id'].values
    drugs_val = X_val['item_id'].values
    drugs_tst = X_tst['item_id'].values

    resistance_tr = X_tr['resistance'].values
    resistance_val = X_val['resistance'].values
    resistance_tst = X_tst['resistance'].values

    print("Training rows:", maldis_tr.shape[0])
    print("Validation rows:", maldis_val.shape[0])
    print("Test rows:", maldis_tst.shape[0])

    # Dataloaders
    loader_tr = DataLoader(
        RecDataset(maldis_tr, meta_tr, drugs_tr, resistance_tr),
        batch_size=batch_size,
        num_workers=0,
        shuffle=True
    )

    loader_val = DataLoader(
        RecDataset(maldis_val, meta_val, drugs_val, resistance_val),
        batch_size=batch_size,
        num_workers=0,
        shuffle=False
    )

    loader_tst = DataLoader(
        RecDataset(maldis_tst, meta_tst, drugs_tst, resistance_tst),
        batch_size=batch_size,
        num_workers=0,
        shuffle=False
    )

    checkpoint_path = os.path.join('Models', f"{filename}_fold{fold}.ckpt")

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        callbacks=[EarlyStopping(monitor="loss_val", mode="min", patience=patience)],
        enable_checkpointing=False,
        logger=False
    )

    if False:
        print("Loading model...")
        model = NCF.load_from_checkpoint(
            checkpoint_path,
            num_feat=maldis_tr.shape[1],
            num_items=num_items,
            num_meta=num_meta,
            weights_only=False,
            sample_encoder=maldi_encoder,
            embedding_dim_samples=embedding_dim_samples,
            embedding_dim_items=embedding_dim_items,
            embedding_dim_metadata=embedding_dim_metadata,
            hidden_dim_samples=hidden_dim_samples
        )

        if os.path.exists(filename + '_scores.pkl'):
            with open(filename + '_scores.pkl', 'rb') as file:
                scores_dict = pickle.load(file)

    else:
        print("Training model...")
        model = NCF(
            num_feat=maldis_tr.shape[1],
            num_items=num_items,
            num_meta=num_meta,
            sample_encoder=maldi_encoder,
            embedding_dim_samples=embedding_dim_samples,
            embedding_dim_items=embedding_dim_items,
            embedding_dim_metadata=embedding_dim_metadata,
            hidden_dim_samples=hidden_dim_samples
        )

        trainer.fit(model, loader_tr, loader_val)

        for category in ['tr', 'val']:
            for metric in metrics:
                metric_name = f'{metric}_{category}'
                if metric_name in trainer.callback_metrics:
                    update_score(metric, fold, category, trainer.callback_metrics[metric_name].cpu().numpy())

        trainer.save_checkpoint(checkpoint_path)

    # Test
    model.eval()
    trainer.test(model, dataloaders=loader_tst, verbose=False)

    for metric in metrics:
        metric_name = f'{metric}_tst'
        if metric_name in trainer.callback_metrics:
            update_score(metric, fold, 'tst', trainer.callback_metrics[metric_name].cpu().numpy())

    # Per-antibiotic AUC
    for d in np.unique(drugs_tst):
        mask_d = (drugs_tst == d)

        with torch.no_grad():
            resistance_pred = model(
                torch.tensor(maldis_tst[mask_d]).float(),
                torch.tensor(meta_tst[mask_d]).long(),
                torch.tensor(drugs_tst[mask_d]).long()
            ).cpu().numpy().ravel()

        if len(np.unique(resistance_tst[mask_d])) > 1:
            antibiotic_auc[d].append(roc_auc_score(resistance_tst[mask_d], resistance_pred))
        else:
            antibiotic_auc[d].append(np.nan)
            print(f'Antibiotic {d} has only one class in test fold {fold}')


# =========================
# AVERAGE AUC PER ANTIBIOTIC
# =========================
average_antibiotic_auc = {
    enc_items.inverse_transform([item])[0]: np.nanmean(auc_scores) if len(auc_scores) > 0 else np.nan
    for item, auc_scores in antibiotic_auc.items()
}


# =========================
# SAVE SCORES
# =========================
with open(filename + '_scores.pkl', 'wb') as file:
    pickle.dump(scores_dict, file)


# =========================
# SCORES DATAFRAME
# =========================
data = []
for metric, folds_dict in scores_dict.items():
    for fold, cat_dict in folds_dict.items():
        for category, score in cat_dict.items():
            data.append((metric, fold, category, score))

scores_df = pd.DataFrame(data, columns=['Metric', 'Fold', 'Category', 'Score'])
scores_df.set_index(['Metric', 'Fold', 'Category'], inplace=True)

print("\nScores dataframe:")
print(scores_df)


# =========================
# GLOBAL TEST AUC
# =========================
auc = [d['tst'] for _, d in scores_dict['auc'].items()]
print(f"\n{maldi_encoder} AUC: {np.nanmean(auc):.3f} +/- {np.nanstd(auc):.3f}")


# =========================
# PER-ANTIBIOTIC AUC
# =========================
print("\nAverage antibiotic AUC:")
print(average_antibiotic_auc)

Number of unique samples: 19265
Number of antibiotics: 32
Number of metadata classes: 1
Total rows: 340244
                                                                   
========== Fold 0 ==========
No data leakage detected.


/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/venv/lib/python3.13/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /export/usuarios01/egarroyo/MALDI_for_AMR_prediction ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name        | Type           | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | embedding_s | Em_Spectrum    | 120    | train | 0    
1 | embedding_d | Embedding      | 480    | train | 0    
2 | embedding_m | Embedding      | 10     | train | 0    
3 | CF_net      | Seque

Training rows: 203908
Validation rows: 68427
Test rows: 67909
Training model...
Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.


/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.


...

Testing DataLoader 0: 100%|██████████| 2139/2139 [00:11<00:00, 190.11it/s]
Antibiotic 29 has only one class in test fold 1

========== Fold 2 ==========
No data leakage detected.


/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/venv/lib/python3.13/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /export/usuarios01/egarroyo/MALDI_for_AMR_prediction ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name        | Type           | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | embedding_s | Em_Spectrum    | 120    | train | 0    
1 | embedding_d | Embedding      | 480    | train | 0    
2 | embedding_m | Embedding      | 10     | train | 0    
3 | CF_net      | Seque

Training rows: 204382
Validation rows: 67921
Test rows: 67941
Training model...
                                                                            

/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.
/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=47` in the `DataLoader` to improve performance.


Epoch 14:  49%|████▉     | 3134/6387 [00:33<00:34, 94.95it/s, acc_val=0.858, auc_val=0.810, acc_tr=0.853, auc_tr=0.803]

In [1]:
import pickle

with open('NCF_metadata_CNN_scores.pkl', 'rb') as f:
    scores_dict = pickle.load(f)

In [2]:
import numpy as np

auc = [d['tst'] for _, d in scores_dict['auc'].items()]
print("Global AUC:", np.mean(auc), "+/-", np.std(auc))

Global AUC: 0.8041172504425049 +/- 0.003131101425095593
